# Hybrid KD T1→S1 — Seed 123

Kết hợp **Sequence KD** và **Dense Mapped Logit-KD** cho cặp
ExpansionNet v2 → MobileNetV3-Large.

- Loss: `0,50 CE_human + 0,25 CE_sequence + 0,25 T² KL_mapped`.
- Teacher: ExpansionNet v2, frozen FP32, ảnh `384×384`.
- Student: S1 MobileNetV3-Large, khởi tạo từ baseline-best seed `123`.
- Temperature: `T=4`.
- Physical/effective batch: `1/64`.
- Chọn checkpoint duy nhất bằng Validation CIDEr-D.
- Held-out Test: `SEALED / NOT USED`.
- Notebook tự lưu `last.pt` trước giới hạn phiên để resume chính xác.

## Kaggle inputs bắt buộc

1. Output `01_MSCOCO_Data_Contract`.
2. Output `02A_Phase2_Shared_Pipeline`.
3. Output baseline S1 đúng seed `123`.
4. Output ExpansionNet v2 preflight và handoff/pseudo-caption.
5. Model source ExpansionNet v2 offline.
6. Output XLKD-02 loss preflight và XLKD-03 training readiness.
7. MS COCO 2014 và `pycocoevalcap==1.2` offline.
8. Khi resume: output `HYB_123` gần nhất.

**Runtime:** GPU T4, Internet Off. Dự kiến `5,3–6,2 ngày lịch/seed`.
Ba seed `42`, `123`, `2026` độc lập và có thể chạy song song.


## Cell 1 — Authorization, integrity và resume seed 123

Xác minh XLKD-04 seed 42, authorization XLKD-05, baseline S1 seed `123`,
production contract bất biến và optional exact-resume checkpoint.


In [1]:
import copy, csv, gc, hashlib, json, math, os, pickle, platform, random, shutil, subprocess, sys, time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 123
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working/HYB_123",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
LOG = OUT / "logs"
RESULT = OUT / "results"
CHECKPOINT = OUT / "checkpoints"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, LOG, RESULT, CHECKPOINT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

_DISCOVERY_INDEX = {}
for _candidate in NOTEBOOK_ROOT.rglob("*"):
    if _candidate.is_file():
        _DISCOVERY_INDEX.setdefault(_candidate.name, []).append(
            _candidate.resolve()
        )

def exactly_one(name):
    matches = sorted(_DISCOVERY_INDEX.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

def verify_csv_manifest(root, reference):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

def find_expansionnet_manifest():
    preferred = INPUT_ROOT / "models"
    search_root = preferred if preferred.is_dir() else INPUT_ROOT
    matches = []
    for candidate in search_root.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, matches
    return matches[0]

if not torch.cuda.is_available():
    raise RuntimeError(
        "XLKD-05 yêu cầu GPU T4 hoặc mạnh hơn. Hãy bật Accelerator rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

X02_PATH = exactly_one("XLKD02_DENSE_MAPPED_LOSS_PREFLIGHT_COMPLETE.json")
AUTH_PATH = exactly_one(
    "XLKD03_DENSE_MAPPED_LOGIT_KD_TRAINING_READINESS_AUTHORIZED.json"
)
X00_AUTH_PATH = exactly_one(
    "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2A_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
S1_PATH = exactly_one("S1_SEED123_FULL_RUN_COMPLETE.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent

X02, AUTH, X00_AUTH, P1, P2A, S1, P3B, P3E = map(
    read_json,
    [X02_PATH, AUTH_PATH, X00_AUTH_PATH, P1_PATH, P2A_PATH,
     S1_PATH, P3B_PATH, P3E_PATH],
)
assert X02["status"] == "complete" and X02["overall_gate"] is True
assert X02["decision"] == "authorize_xlkd03_training_readiness"
assert X02["full_training_authorized"] is False
assert AUTH["status"] == "authorized"
assert AUTH["required_next"].startswith("XLKD-03")
assert AUTH["full_training_authorized"] is False
assert float(AUTH["temperature"]) == 4.0
assert float(AUTH["alpha_human_ce"]) == 0.5
assert float(AUTH["alpha_kd"]) == 0.5
assert int(AUTH["physical_batch_size"]) == 1
assert X00_AUTH["status"] == "authorized"
assert P1["status"] == "ready"
assert P2A["status"] == "ready"
assert S1["status"] == "complete"
assert S1["student_id"] == "S1" and int(S1["seed"]) == 123
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete"
for marker in [X02, AUTH, X00_AUTH, S1, P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2A["data_policy"]["test_set_status"] == "sealed"
assert P2A["data_policy"]["test_set_used"] is False

x02_rows = verify_csv_manifest(X02_PATH.parent, X02["manifest"]["csv"])
assert len(x02_rows) == 11
p2a_rows = verify_csv_manifest(P2A_PATH.parent, P2A["manifests"]["csv"])
assert len(p2a_rows) == 38

decision = read_json(X02_PATH.parent / "results" / "xlkd02_decision.json")
assert decision["overall_gate"] is True
assert all(decision["gates"].values())
LOSS_CONTRACT_PATH = (
    X02_PATH.parent / "configs" / "xlkd02_executed_loss_contract.json"
).resolve()
LOSS_CONTRACT = read_json(LOSS_CONTRACT_PATH)
assert LOSS_CONTRACT["status"] == "locked"
assert LOSS_CONTRACT["temperature"] == 4.0
assert LOSS_CONTRACT["full_training_authorized"] is False

mapping_ref = X00_AUTH["mapping_contract"]
MAPPING_PATH = resolve_inside(X00_AUTH_PATH.parent, mapping_ref["relative_path"])
assert sha256(MAPPING_PATH) == mapping_ref["sha256"]
MAPPING_PAYLOAD = read_json(MAPPING_PATH)
mapping = {
    int(source): int(target)
    for source, target in MAPPING_PAYLOAD["mapping"].items()
}
assert len(mapping) == 9959
assert MAPPING_PAYLOAD["mapping_collisions"] == 0

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["model_id"] == "expansionnet_v2"

def verify_model_record(record):
    target = resolve_inside(MODEL_ROOT, record["relative_path"])
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_model_record(record)
for record in model_manifest["integration_patches"]:
    target = resolve_inside(MODEL_ROOT, record["file"])
    assert target.is_file() and sha256(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_tokens = pickle.load(handle)
NATIVE_WORD2IDX = native_tokens["word2idx_dict"]
NATIVE_IDX2WORD = native_tokens["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[native_tokens["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[native_tokens["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (78, 79, 77, 80)

PSEUDO_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert sha256(PSEUDO_PATH) == PSEUDO_SUMMARY["jsonl_sha256"]
assert PSEUDO_SUMMARY["validation_overlap"] == 0
assert PSEUDO_SUMMARY["held_out_test_overlap"] == 0

BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed123_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")

write_json(AUDIT / "xlkd03_cell01_authorization.json", {
    "schema_version": "xlkd03_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd02_manifest_rows": len(x02_rows),
    "phase2a_manifest_rows": len(p2a_rows),
    "xlkd02_gates": decision["gates"],
    "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "student_checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
    "mapping_tokens": len(mapping),
    "pseudo_caption_rows": 113287,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "strict_deterministic": True,
    "checkpoint_payload_read": False,
    "dataset_read": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY")
print("=" * 78)
print("Status                    : PASSED")
print("XLKD-02 manifest          : 11 / 11 VERIFIED")
print("Phase 02A manifest        : 38 / 38 VERIFIED")
print("XLKD-02 gates             : 9 / 9 PASSED")
print("Vocabulary mapping        : 9,959 VERIFIED")
print("Pseudo-caption corpus     : 113,287 VERIFIED")
print("Teacher / Student ckpt    : VERIFIED / VERIFIED")
print("Validation / Test         : NOT READ / SEALED")
print("Full training             : NOT STARTED")
print("Progress                  : 1 / 7 code cells")


# XLKD-05 confirmation may start only after the locked seed-42 run.
X03_COMPLETE_PATH = exactly_one(
    "XLKD03_DENSE_MAPPED_TRAINING_READINESS_COMPLETE.json"
)
X04_AUTH_PATH = exactly_one(
    "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_AUTHORIZED.json"
)
DEVELOPMENT_COMPLETE_PATH = exactly_one(
    "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json"
)
X05_AUTH_PATH = exactly_one(
    "XLKD05_DENSE_MAPPED_LOGIT_KD_CONFIRMATION_SEEDS_AUTHORIZED.json"
)
X03_COMPLETE = read_json(X03_COMPLETE_PATH)
X04_AUTH = read_json(X04_AUTH_PATH)
DEVELOPMENT_COMPLETE = read_json(DEVELOPMENT_COMPLETE_PATH)
X05_AUTH = read_json(X05_AUTH_PATH)

assert X03_COMPLETE["status"] == "complete"
assert X03_COMPLETE["production_data_pipeline"] == "passed"
assert X03_COMPLETE["model_optimizer_engine"] == "passed"
assert X03_COMPLETE["validation_policy_inheritance"] == "passed"
assert X03_COMPLETE["discarded_pilot"] == "passed"
assert X03_COMPLETE["production_contract"] == "locked"
assert X03_COMPLETE["xlkd04_seed42_authorized"] is True
assert X04_AUTH["status"] == "authorized"
assert int(X04_AUTH["seed"]) == 42
assert X04_AUTH["full_training_authorized"] is True
assert X04_AUTH["validation_selection_authorized"] is True
assert X04_AUTH["test_set_status"] == "sealed"
assert X04_AUTH["test_set_used"] is False

assert DEVELOPMENT_COMPLETE["status"] == "complete"
assert DEVELOPMENT_COMPLETE["experiment_id"] == (
    "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGITKD"
)
assert int(DEVELOPMENT_COMPLETE["seed"]) == 42
assert DEVELOPMENT_COMPLETE["confirmation_seeds_authorized"] is True
assert DEVELOPMENT_COMPLETE["manifest_verification"] == "all_passed"
assert DEVELOPMENT_COMPLETE["test_set_status"] == "sealed"
assert DEVELOPMENT_COMPLETE["test_set_used"] is False

assert X05_AUTH["status"] == "authorized"
assert X05_AUTH["experiment_id"] == (
    "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGITKD"
)
assert int(X05_AUTH["development_seed_complete"]) == 42
assert SEED in [int(value) for value in X05_AUTH["confirmation_seeds"]]
assert X05_AUTH["method_changes_allowed"] is False
assert X05_AUTH["required_initialization"] == (
    "matching S1 baseline-best checkpoint"
)
assert X05_AUTH["test_set_status"] == "sealed"
assert X05_AUTH["test_set_used"] is False

x03_rows = verify_csv_manifest(
    X03_COMPLETE_PATH.parent, X03_COMPLETE["manifest"]["csv"]
)
assert len(x03_rows) == 12
development_rows = verify_csv_manifest(
    DEVELOPMENT_COMPLETE_PATH.parent,
    DEVELOPMENT_COMPLETE["manifest"]["csv"],
)
assert len(development_rows) == int(
    DEVELOPMENT_COMPLETE["manifest"]["csv"]["rows"]
)
assert len(development_rows) == 55

contract_ref = X04_AUTH["production_contract"]
PRODUCTION_CONTRACT_PATH = resolve_inside(
    X04_AUTH_PATH.parent, contract_ref["relative_path"]
)
assert sha256(PRODUCTION_CONTRACT_PATH) == contract_ref["sha256"]
PRODUCTION_CONTRACT = read_json(PRODUCTION_CONTRACT_PATH)
assert PRODUCTION_CONTRACT["status"] == "locked"
assert PRODUCTION_CONTRACT["loss"]["temperature"] == 4.0
assert PRODUCTION_CONTRACT["loss"]["alpha_human_ce"] == 0.5
assert PRODUCTION_CONTRACT["loss"]["alpha_kd"] == 0.5
assert PRODUCTION_CONTRACT["training"]["physical_batch"] == 1
assert PRODUCTION_CONTRACT["training"]["gradient_accumulation"] == 64
assert PRODUCTION_CONTRACT["validation"]["selection_primary"] == "CIDEr-D"
assert PRODUCTION_CONTRACT["test_set_status"] == "sealed"
assert PRODUCTION_CONTRACT["test_set_used"] is False
assert DEVELOPMENT_COMPLETE["production_contract"]["source_sha256"] == (
    sha256(PRODUCTION_CONTRACT_PATH)
)

roadmap_ref = X04_AUTH["roadmap"]
FINAL_ROADMAP_PATH = resolve_inside(
    X04_AUTH_PATH.parent, roadmap_ref["relative_path"]
)
assert sha256(FINAL_ROADMAP_PATH) == roadmap_ref["sha256"]

resume_matches = sorted(
    _DISCOVERY_INDEX.get("hyb_s1_seed123_last.pt", [])
)
assert len(resume_matches) <= 1, resume_matches
INPUT_RESUME_CHECKPOINT = (
    resume_matches[0] if resume_matches else None
)

# Compatibility names used by the already-audited Validation/full-run engine.
OUTPUT_ROOT = OUT
AUDIT_DIR = AUDIT
CONFIG_DIR = CONFIG
LOG_DIR = LOG
RESULT_DIR = RESULT
CHECKPOINT_DIR = CHECKPOINT
MANIFEST_DIR = MANIFEST
VALIDATION_DIR = OUT / "validation"
for directory in [VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
now_utc = now
sha256_file = sha256
PSEUDO_CORPUS = PSEUDO_PATH

BASELINE_CIDER = float(S1["best_validation_metrics"]["CIDEr-D"])
BASELINE_BLEU4 = float(S1["best_validation_metrics"]["BLEU-4"])
BASELINE_CE = float(S1["best_validation_cross_entropy"])
BASELINE_METRICS = {
    key: float(value)
    for key, value in S1["best_validation_metrics"].items()
}

write_json(AUDIT / "xlkd05_cell01_authorization.json", {
    "schema_version": "xlkd05_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd03_manifest_rows": len(x03_rows),
    "xlkd04_manifest_rows": len(development_rows),
    "development_seed42_complete": True,
    "confirmation_authorization": "verified",
    "method_changes_allowed": False,
    "production_contract_sha256": sha256(PRODUCTION_CONTRACT_PATH),
    "roadmap_sha256": sha256(FINAL_ROADMAP_PATH),
    "seed": SEED,
    "resume_checkpoint_found": INPUT_RESUME_CHECKPOINT is not None,
    "validation_authorized": True,
    "full_training_authorized": True,
    "confirmation_seed_authorized": True,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("XLKD-05 CELL 1 — FULL-RUN AUTHORIZATION")
print("=" * 78)
print("Status                    : PASSED")
print("XLKD-03 manifest          : 12 / 12 VERIFIED")
print("XLKD-04 manifest          : 55 / 55 VERIFIED")
print("Development seed 42       : COMPLETE")
print("Production contract       : LOCKED")
print("Seed                      :", SEED)
print("Resume checkpoint found   :", INPUT_RESUME_CHECKPOINT is not None)
print("Validation                : AUTHORIZED")
print("Full training             : AUTHORIZED")
print("Confirmation seed         : AUTHORIZED")
print("Test Set status           : SEALED")


XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY
Status                    : PASSED
XLKD-02 manifest          : 11 / 11 VERIFIED
Phase 02A manifest        : 38 / 38 VERIFIED
XLKD-02 gates             : 9 / 9 PASSED
Vocabulary mapping        : 9,959 VERIFIED
Pseudo-caption corpus     : 113,287 VERIFIED
Teacher / Student ckpt    : VERIFIED / VERIFIED
Validation / Test         : NOT READ / SEALED
Full training             : NOT STARTED
Progress                  : 1 / 7 code cells
XLKD-05 CELL 1 — FULL-RUN AUTHORIZATION
Status                    : PASSED
XLKD-03 manifest          : 12 / 12 VERIFIED
XLKD-04 manifest          : 55 / 55 VERIFIED
Development seed 42       : COMPLETE
Production contract       : LOCKED
Seed                      : 123
Resume checkpoint found   : True
Validation                : AUTHORIZED
Full training             : AUTHORIZED
Confirmation seed         : AUTHORIZED
Test Set status           : SEALED


## Cell 2 — Production Train+Restval và authorized Validation


In [2]:
STARTED = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision.transforms import InterpolationMode
from torchvision.transforms import RandomResizedCrop
from torchvision.transforms import functional as TF

TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "train")],
)
TRAIN_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    TRAIN_PATHS[image_id] = (leaf / Path(row["image_path"]).name).resolve()
assert len(TRAIN_PATHS) == 113287

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append((
            int(row["annotation_id"]),
            tuple(int(value) for value in row["token_ids"]),
        ))
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

PSEUDO_BY_IMAGE = {}
with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        teacher_ids = tuple(int(value) for value in row["teacher_token_ids"])
        student_ids = tuple(int(value) for value in row["student_token_ids"])
        assert len(teacher_ids) == len(student_ids)
        assert [mapping[value] for value in teacher_ids] == list(student_ids)
        assert image_id not in PSEUDO_BY_IMAGE
        PSEUDO_BY_IMAGE[image_id] = (teacher_ids, student_ids)
assert set(PSEUDO_BY_IMAGE) == set(TRAIN_PATHS)
TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|xlkd04"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)

def dual_train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = RandomResizedCrop.get_params(
            image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
        )
        do_flip = bool(torch.rand(()) < 0.5)
    teacher = TF.resized_crop(
        image, top, left, height, width, [384, 384],
        InterpolationMode.BILINEAR, antialias=True,
    )
    student = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        teacher, student = TF.hflip(teacher), TF.hflip(student)
    teacher = TF.normalize(TF.to_tensor(teacher), IMAGENET_MEAN, IMAGENET_STD)
    student = TF.normalize(TF.to_tensor(student), IMAGENET_MEAN, IMAGENET_STD)
    return teacher, student

class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        teacher_ids, student_ids = PSEUDO_BY_IMAGE[image_id]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            teacher_image, student_image = dual_train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "teacher_ids": torch.tensor(teacher_ids, dtype=torch.long),
            "student_pseudo_ids": torch.tensor(student_ids, dtype=torch.long),
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
        }

class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(self.length, generator=generator).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor

def production_collate(rows):
    assert len(rows) == 1
    row = rows[0]
    return {
        "image_ids": torch.tensor([row["image_id"]], dtype=torch.long),
        "annotation_ids": torch.tensor([row["annotation_id"]], dtype=torch.long),
        "teacher_image": row["teacher_image"].unsqueeze(0),
        "student_image": row["student_image"].unsqueeze(0),
        "teacher_ids": row["teacher_ids"].unsqueeze(0),
        "student_pseudo_ids": row["student_pseudo_ids"].unsqueeze(0),
        "human_ids": row["human_ids"].unsqueeze(0),
    }

TRAIN_DATASET = ProductionTrainDataset()

def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=1,
        sampler=EpochCursorSampler(len(TRAIN_DATASET), epoch, cursor),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=production_collate,
    )

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "teacher_image", "student_image",
    "teacher_ids", "student_pseudo_ids", "human_ids",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert batch_a["teacher_image"].shape == (1, 3, 384, 384)
assert batch_a["student_image"].shape == (1, 3, 224, 224)
assert torch.isfinite(batch_a["teacher_image"]).all()
assert torch.isfinite(batch_a["student_image"]).all()

write_json(AUDIT / "xlkd05_cell02_data_pipeline.json", {
    "schema_version": "xlkd05_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "train_images_per_epoch": len(TRAIN_DATASET),
    "human_caption_sampling": "one per image per epoch, cyclic",
    "teacher_sequence_per_image": 1,
    "physical_batch_size": 1,
    "shared_stateless_geometry": True,
    "teacher_resolution": [384, 384],
    "student_resolution": [224, 224],
    "teacher_student_sequence_alignment": "113287 / 113287",
    "two_loader_first_batch_exact": True,
    "resume_cursor_supported": True,
    "validation_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE")
print("=" * 78)
print("Status                    : PASSED")
print("Train images / epoch      : 113,287")
print("Human caption / image     : 1 CYCLIC")
print("Teacher sequence / image  : 1 FIXED")
print("Sequence alignment        : 113,287 / 113,287")
print("Physical batch            : 1")
print("Teacher / Student input   : 384 / 224")
print("Stateless shared geometry : True")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 7 code cells")


VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
validation_path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "validation")],
)
VALIDATION_PATHS = {}
for row in validation_path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    VALIDATION_PATHS[image_id] = (
        leaf / Path(row["image_path"]).name
    ).resolve()
assert len(VALIDATION_PATHS) == 5000

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for arrow_batch in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in arrow_batch.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append((
            int(row["annotation_id"]),
            image_id,
            tuple(int(value) for value in row["token_ids"]),
        ))
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))

def validation_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    tensor = TF.to_tensor(TF.center_crop(image, [224, 224]))
    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)

class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }

class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {"image_id": image_id, "student_image": image}

def pad_sequences(sequences):
    maximum = max(item.numel() for item in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, item in enumerate(sequences):
        result[index, :item.numel()] = item
    return result

def validation_caption_collate(rows):
    full = pad_sequences([row["token_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
        "decoder_attention_mask": full[:, :-1].ne(0),
    }

def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }

VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()
VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=validation_caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

assert set(VALIDATION_PATHS).isdisjoint(TRAIN_PATHS)
write_json(AUDIT / "xlkd05_cell02_validation_data.json", {
    "schema_version": "xlkd05_cell02_validation_data_v1",
    "created_at_utc": now(),
    "status": "passed",
    "training_images": len(TRAIN_DATASET),
    "validation_images": len(VALIDATION_IMAGE_DATASET),
    "validation_captions": len(VALIDATION_CAPTION_DATASET),
    "validation_overlap_with_training": 0,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Validation images         : 5,000")
print("Validation references     : 25,010")
print("Train/Validation overlap  : 0")
print("Test Set status           : SEALED")


XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE
Status                    : PASSED
Train images / epoch      : 113,287
Human caption / image     : 1 CYCLIC
Teacher sequence / image  : 1 FIXED
Sequence alignment        : 113,287 / 113,287
Physical batch            : 1
Teacher / Student input   : 384 / 224
Stateless shared geometry : True
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 7 code cells
Validation images         : 5,000
Validation references     : 25,010
Train/Validation overlap  : 0
Test Set status           : SEALED


## Cell 3 — Frozen Teacher, S1 baseline-best và mapped Logit-KD


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048, num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
teacher_payload = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    teacher_payload["model_state_dict"], strict=True
)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
del teacher_payload
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
assert all(not parameter.requires_grad for parameter in TEACHER.parameters())

DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )

STUDENT_ENCODER = S1Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed123_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
baseline_payload = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert baseline_payload["schema_version"] == (
    "phase2_s1_full_training_checkpoint_v1"
)
STUDENT_ENCODER.load_state_dict(
    baseline_payload["encoder_state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    baseline_payload["decoder_state_dict"], strict=True
)
BASE_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
BASE_DECODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
del baseline_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3231024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328
assert sum(p.numel() for p in TEACHER.parameters()) > 100_000_000

mapped_src = torch.tensor(
    sorted(mapping), dtype=torch.long, device=device
)
mapped_dst = torch.tensor(
    [mapping[index] for index in sorted(mapping)],
    dtype=torch.long, device=device,
)
assert mapped_src.numel() == mapped_dst.numel() == 9959

def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 46

def reset_student_baseline():
    STUDENT_ENCODER.load_state_dict(BASE_ENCODER_STATE, strict=True)
    STUDENT_DECODER.load_state_dict(BASE_DECODER_STATE, strict=True)
    for parameter in list(STUDENT_ENCODER.parameters()) + list(STUDENT_DECODER.parameters()):
        parameter.grad = None

def teacher_mapped_distribution(batch, temperature=4.0):
    images = batch["teacher_image"].to(device, non_blocking=True)
    full = batch["teacher_ids"].to(device, non_blocking=True)
    decoder_input = full[:, :-1]
    labels = full[:, 1:]
    num_pads = (decoder_input == NATIVE_PAD).sum(dim=1).tolist()
    # Do not use inference_mode here: q is consumed by F.kl_div while
    # autograd records the Student branch. A tensor created in
    # inference_mode cannot be saved for that backward pass. no_grad keeps
    # the Teacher frozen but returns ordinary tensors that are valid targets.
    with torch.no_grad():
        logits = TEACHER(
            enc_x=images,
            dec_x=decoder_input,
            enc_x_num_pads=[0] * images.shape[0],
            dec_x_num_pads=[int(value) for value in num_pads],
            apply_log_softmax=False,
            mode="forward",
        )
        assert logits.shape == (
            images.shape[0], decoder_input.shape[1], 10000
        )
        probabilities = torch.softmax(logits.float() / temperature, dim=-1)
        selected = probabilities.index_select(-1, mapped_src)
        dense = torch.zeros(
            *selected.shape[:-1], 10080,
            dtype=selected.dtype, device=device,
        )
        index = mapped_dst.view(1, 1, -1).expand_as(selected)
        # XLKD-00 proved that the map is collision-free, so scatter (not
        # scatter_add) is mathematically equivalent and deterministic.
        dense.scatter_(-1, index, selected)
        mass = dense.sum(dim=-1)
        q = dense / mass.unsqueeze(-1)
        mask = labels.ne(NATIVE_PAD)
    return q.detach(), mask.detach(), mass.detach()


STARTED_ENGINE = time.perf_counter()

TEMPERATURE = 4.0
ALPHA_HUMAN = 0.50
ALPHA_SEQUENCE = 0.25
ALPHA_KD = 0.25
assert abs(ALPHA_HUMAN + ALPHA_SEQUENCE + ALPHA_KD - 1.0) < 1e-12
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 64
EFFECTIVE_BATCH = 64
STEPS_PER_EPOCH = math.ceil(113287 / EFFECTIVE_BATCH)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())
all_student_parameters = (
    backbone_parameters + adapter_parameters + decoder_parameters
)

def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4,
    )
    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / max(1, WARMUP_STEPS)
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler

def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }

def optimizer_step(microbatches, optimizer, scheduler, scaler, retry_limit=10):
    human_total = sum(
        int(batch["human_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    kd_total = sum(
        int(batch["student_pseudo_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0 and kd_total > 0
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_value = 0.0
        sequence_sum_value = 0.0
        kd_sum_value = 0.0
        mapped_values = []
        for cpu_batch in microbatches:
            q_teacher, kd_mask, mapped_mass = (
                teacher_mapped_distribution(
                    cpu_batch, temperature=TEMPERATURE
                )
            )
            current = to_device(cpu_batch)
            human = current["human_ids"]
            pseudo = current["student_pseudo_ids"]
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = STUDENT_ENCODER(current["student_image"])
                human_logits = STUDENT_DECODER(
                    human[:, :-1], memory, human[:, :-1].ne(0)
                )
                pseudo_logits = STUDENT_DECODER(
                    pseudo[:, :-1], memory, pseudo[:, :-1].ne(0)
                )
                human_sum = F.cross_entropy(
                    human_logits.float().reshape(-1, 10080),
                    human[:, 1:].reshape(-1),
                    ignore_index=0, reduction="sum",
                )
                sequence_sum = F.cross_entropy(
                    pseudo_logits.float().reshape(-1, 10080),
                    pseudo[:, 1:].reshape(-1),
                    ignore_index=0, reduction="sum",
                )
                student_log_prob = F.log_softmax(
                    pseudo_logits.float() / TEMPERATURE, dim=-1
                )
                assert torch.equal(kd_mask, pseudo[:, 1:].ne(0))
                kd_sum = (
                    F.kl_div(
                        student_log_prob, q_teacher, reduction="none"
                    ).sum(dim=-1)[kd_mask].sum()
                    * (TEMPERATURE ** 2)
                )
                normalized = (
                    ALPHA_HUMAN * human_sum / human_total
                    + ALPHA_SEQUENCE * sequence_sum / kd_total
                    + ALPHA_KD * kd_sum / kd_total
                )
            scaler.scale(normalized / BACKWARD_DIVISOR).backward()
            human_sum_value += float(human_sum.detach())
            sequence_sum_value += float(sequence_sum.detach())
            kd_sum_value += float(kd_sum.detach())
            mapped_values.extend(
                mapped_mass[kd_mask].detach().cpu().double().tolist()
            )
            del (
                current, memory, human_logits, pseudo_logits,
                q_teacher, kd_mask, mapped_mass, student_log_prob,
                human_sum, sequence_sum, kd_sum, normalized,
            )
        scaler.unscale_(optimizer)
        for parameter in all_student_parameters:
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            all_student_parameters, 1.0
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        scheduler.step()
        human_ce = human_sum_value / human_total
        sequence_ce = sequence_sum_value / kd_total
        kd_kl_t2 = kd_sum_value / kd_total
        return {
            "combined_loss": (
                ALPHA_HUMAN * human_ce
                + ALPHA_SEQUENCE * sequence_ce
                + ALPHA_KD * kd_kl_t2
            ),
            "human_ce": human_ce,
            "sequence_ce": sequence_ce,
            "mapped_kl_t2": kd_kl_t2,
            "human_tokens": human_total,
            "kd_positions": kd_total,
            "mapped_mass_mean": float(np.mean(mapped_values)),
            "mapped_mass_minimum": float(np.min(mapped_values)),
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }

optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771
assert len(optimizer.param_groups) == 3

write_json(CONFIG / "xlkd05_optimizer_contract.json", {
    "schema_version": "xlkd05_optimizer_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "optimizer": "AdamW",
    "learning_rates": {
        "backbone": 1e-4,
        "adapter": 3e-4,
        "decoder": 3e-4,
    },
    "weight_decay": 1e-4,
    "physical_batch": 1,
    "accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "maximum_epochs": MAX_EPOCHS,
    "total_steps": TOTAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "minimum_lr_factor": 0.05,
    "mixed_precision": "Student FP16 autocast plus GradScaler; Teacher FP32 no-grad",
    "gradient_clip": 1.0,
    "overflow_retry_limit": 10,
    "backward_divisor": BACKWARD_DIVISOR,
    "temperature": TEMPERATURE,
    "alpha_human_ce": ALPHA_HUMAN,
    "alpha_sequence_ce": ALPHA_SEQUENCE,
    "alpha_kd": ALPHA_KD,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd05_cell03_model_optimizer.json", {
    "schema_version": "xlkd05_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "teacher_strict_load": True,
    "teacher_frozen": True,
    "student_strict_load": True,
    "student_initialization": "S1 seed-123 baseline best",
    "student_parameters": 8990352,
    "optimizer_parameter_groups": 3,
    "physical_batch": 1,
    "accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "total_steps": TOTAL_STEPS,
    "scheduler_advanced": False,
    "optimizer_step_performed": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED_ENGINE,
})
print("=" * 78)
print("XLKD-03 CELL 3 — MODEL AND OPTIMIZER ENGINE")
print("=" * 78)
print("Status                    : PASSED")
print("Teacher                   : STRICT-LOAD / FROZEN / FP32")
print("Student init              : S1 SEED-123 BASELINE BEST")
print("Student parameters        : 8,990,352")
print("Optimizer groups          : 3 / 3")
print("Physical/accum/effective  : 1 / 64 / 64")
print("Steps / epoch / total     :", STEPS_PER_EPOCH, "/", TOTAL_STEPS)
print("Student AMP / GradScaler  : ENABLED / 1024")
print("Optimizer step            : NOT PERFORMED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 3 / 7 code cells")


def student_logits(batch):
    memory = STUDENT_ENCODER(batch["student_images"])
    return STUDENT_DECODER(
        batch["decoder_input_ids"],
        memory,
        batch["decoder_attention_mask"],
    )

# Compatibility name for the inherited full-run summary expression.
ALPHA = ALPHA_KD
assert (ALPHA_HUMAN, ALPHA_SEQUENCE, ALPHA_KD) == (0.50, 0.25, 0.25)
assert TEMPERATURE == 4.0


XLKD-03 CELL 3 — MODEL AND OPTIMIZER ENGINE
Status                    : PASSED
Teacher                   : STRICT-LOAD / FROZEN / FP32
Student init              : S1 SEED-123 BASELINE BEST
Student parameters        : 8,990,352
Optimizer groups          : 3 / 3
Physical/accum/effective  : 1 / 64 / 64
Steps / epoch / total     : 1771 / 35420
Student AMP / GradScaler  : ENABLED / 1024
Optimizer step            : NOT PERFORMED
Validation / Test         : NOT READ / SEALED
Progress                  : 3 / 7 code cells


## Cell 4 — Full Validation metrics và batched Beam-3


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "xlkd05_cell04_validation_engine.json",
    {
        "audit_version": "xlkd05_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("XLKD-05 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : atomic checkpoint and production pilot")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 18467.77 tokens per second.
PTBTokenizer tokenized 385 tokens at 6965.60 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
XLKD-05 CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 2.2743104368775757
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : atomic checkpoint and production pilot


## Cell 5 — Full training với atomic exact resume


In [5]:
CELL_STARTED_AT = time.perf_counter()
from torch.torch_version import TorchVersion


LAST_CHECKPOINT = (
    CHECKPOINT_DIR / "hyb_s1_seed123_last.pt"
)
BEST_CHECKPOINT = (
    CHECKPOINT_DIR / "hyb_s1_seed123_best.pt"
)
STEP_LOG = LOG_DIR / "hyb_s1_seed123_steps.jsonl"
EPOCH_HISTORY_CSV = (
    LOG_DIR / "hyb_s1_seed123_epoch_history.csv"
)
RUN_STATE_PATH = OUTPUT_ROOT / "HYB_RUN_STATE.json"

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3
VOCABULARY_SIZE = 10080
BEAM_SIZE = 3
LENGTH_PENALTY = 0.7
MAXIMUM_DECODER_STEPS = 31
MINIMUM_LEXICAL_TOKENS = 2
FORBIDDEN_GENERATION_TOKEN_IDS = (PAD_ID, BOS_ID, UNK_ID)


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path, optimizer, scheduler, scaler, progress,
    history, best_result, no_improve_epochs,
):
    atomic_save(
        path,
        {
            "schema_version": "hybrid_checkpoint_v1",
            "experiment_id": "EXPANSIONNET_S1_HYBRID_KD",
            "seed": SEED,
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "history": history,
            "best_result": best_result,
            "no_improve_epochs": no_improve_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
            "production_contract_sha256": sha256_file(
                PRODUCTION_CONTRACT_PATH
            ),
            "temperature": TEMPERATURE,
            "alpha_human_ce": ALPHA_HUMAN,
            "alpha_sequence_ce": ALPHA_SEQUENCE,
            "alpha_kd": ALPHA_KD,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


def load_checkpoint(path, optimizer, scheduler, scaler):
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
    assert checkpoint["schema_version"] == (
        "hybrid_checkpoint_v1"
    )
    assert checkpoint["experiment_id"] == "EXPANSIONNET_S1_HYBRID_KD"
    assert int(checkpoint["seed"]) == 123
    assert checkpoint["test_set_status"] == "sealed"
    assert checkpoint["test_set_used"] is False
    assert checkpoint["pseudo_corpus_sha256"] == (
        sha256_file(PSEUDO_CORPUS)
    )
    assert checkpoint["production_contract_sha256"] == (
        sha256_file(PRODUCTION_CONTRACT_PATH)
    )
    STUDENT_ENCODER.load_state_dict(
        checkpoint["student_encoder_state"], strict=True
    )
    STUDENT_DECODER.load_state_dict(
        checkpoint["student_decoder_state"], strict=True
    )
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])
    torch.set_rng_state(checkpoint["cpu_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    return (
        checkpoint["progress"],
        checkpoint["history"],
        checkpoint["best_result"],
        int(checkpoint["no_improve_epochs"]),
    )


@torch.inference_mode()
def batched_beam_search(visual_memory):
    batch_size, memory_length, memory_dimension = visual_memory.shape
    assert memory_length == 49 and memory_dimension == 256
    beam_tokens = torch.full(
        (batch_size, BEAM_SIZE, 1),
        BOS_ID, dtype=torch.long, device=device,
    )
    beam_scores = torch.full(
        (batch_size, BEAM_SIZE),
        float("-inf"), dtype=torch.float32, device=device,
    )
    beam_scores[:, 0] = 0.0
    beam_finished = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.bool, device=device
    )
    beam_lengths = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.long, device=device
    )
    expanded_memory = (
        visual_memory.unsqueeze(1)
        .expand(-1, BEAM_SIZE, -1, -1)
        .reshape(batch_size * BEAM_SIZE, memory_length, memory_dimension)
        .contiguous()
    )
    for generation_step in range(MAXIMUM_DECODER_STEPS):
        flat_tokens = beam_tokens.reshape(
            batch_size * BEAM_SIZE, -1
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = STUDENT_DECODER(
                flat_tokens,
                expanded_memory,
                flat_tokens.ne(PAD_ID),
            )
        log_probs = F.log_softmax(
            logits[:, -1, :].float(), dim=-1
        ).reshape(batch_size, BEAM_SIZE, VOCABULARY_SIZE)
        log_probs[:, :, list(FORBIDDEN_GENERATION_TOKEN_IDS)] = (
            float("-inf")
        )
        if generation_step < MINIMUM_LEXICAL_TOKENS:
            log_probs[:, :, EOS_ID] = float("-inf")
        if beam_finished.any():
            log_probs = log_probs.masked_fill(
                beam_finished.unsqueeze(-1), float("-inf")
            )
            finished_indices = beam_finished.nonzero(as_tuple=False)
            log_probs[
                finished_indices[:, 0],
                finished_indices[:, 1],
                EOS_ID,
            ] = 0.0
        candidate_scores = beam_scores.unsqueeze(-1) + log_probs
        next_scores, flat_indices = torch.topk(
            candidate_scores.reshape(
                batch_size, BEAM_SIZE * VOCABULARY_SIZE
            ),
            k=BEAM_SIZE,
            dim=-1,
        )
        parent = flat_indices // VOCABULARY_SIZE
        next_tokens = flat_indices % VOCABULARY_SIZE
        parent_tokens = torch.gather(
            beam_tokens,
            1,
            parent.unsqueeze(-1).expand(
                -1, -1, beam_tokens.shape[-1]
            ),
        )
        parent_finished = torch.gather(
            beam_finished, 1, parent
        )
        parent_lengths = torch.gather(
            beam_lengths, 1, parent
        )
        beam_tokens = torch.cat(
            [parent_tokens, next_tokens.unsqueeze(-1)], dim=-1
        )
        beam_scores = next_scores
        beam_lengths = parent_lengths + (~parent_finished).long()
        beam_finished = parent_finished | (next_tokens == EOS_ID)
        if bool(beam_finished.all()):
            break
    normalized = beam_scores / (
        beam_lengths.clamp_min(1).float() ** LENGTH_PENALTY
    )
    best = normalized.argmax(dim=1)
    batch_indices = torch.arange(batch_size, device=device)
    return {
        "token_ids": beam_tokens[batch_indices, best].cpu(),
        "raw_scores": beam_scores[batch_indices, best].cpu(),
        "normalized_scores": normalized[batch_indices, best].cpu(),
    }


@torch.inference_mode()
def generate_validation_captions():
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, cpu_batch in enumerate(
        VALIDATION_IMAGE_LOADER
    ):
        image_ids = [
            int(value) for value in cpu_batch["image_ids"].tolist()
        ]
        images = cpu_batch["student_images"].to(
            device, non_blocking=True
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = STUDENT_ENCODER(images)
        beam = batched_beam_search(memory)
        for row_index, image_id in enumerate(image_ids):
            token_ids = [
                int(value)
                for value in beam["token_ids"][row_index].tolist()
            ]
            caption = decode_tokens(token_ids)
            assert caption
            assert image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "caption": caption,
                "token_ids": token_ids,
                "raw_score": float(
                    beam["raw_scores"][row_index]
                ),
                "normalized_score": float(
                    beam["normalized_scores"][row_index]
                ),
            }
        if (batch_index + 1) % 25 == 0:
            print(
                f"Generated {len(hypotheses):,}/5,000 "
                "Validation captions"
            )
    assert len(hypotheses) == 5000
    assert set(hypotheses) == set(VALIDATION_REFERENCES)
    return hypotheses, records, time.perf_counter() - started


def evaluate_validation(epoch):
    ce, valid_tokens = teacher_forced_ce(
        VALIDATION_CAPTION_LOADER
    )
    hypotheses, records, generation_seconds = (
        generate_validation_captions()
    )
    metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, hypotheses
    )
    result = {
        "epoch": int(epoch),
        "teacher_forced_ce": float(ce),
        "valid_tokens": int(valid_tokens),
        "metrics": metrics,
        "prediction_count": len(hypotheses),
        "generation_seconds": generation_seconds,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_validation.json",
        result,
    )
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_predictions.json",
        {
            str(image_id): record
            for image_id, record in records.items()
        },
    )
    return result


def selection_key(result):
    return (
        float(result["metrics"]["CIDEr-D"]),
        float(result["metrics"]["BLEU-4"]),
        -float(result["teacher_forced_ce"]),
        -int(result["epoch"]),
    )


def append_step_log(row):
    with STEP_LOG.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )


def write_epoch_history(history):
    if not history:
        return
    rows = []
    for result in history:
        rows.append(
            {
                "epoch": result["epoch"],
                "train_combined_loss": result["train_combined_loss"],
                "train_human_ce": result["train_human_ce"],
                "train_sequence_ce": result["train_sequence_ce"],
                "train_mapped_kl_t2": result[
                    "train_mapped_kl_t2"
                ],
                "validation_ce": result["teacher_forced_ce"],
                "BLEU-1": result["metrics"]["BLEU-1"],
                "BLEU-4": result["metrics"]["BLEU-4"],
                "METEOR": result["metrics"]["METEOR"],
                "ROUGE-L": result["metrics"]["ROUGE-L"],
                "CIDEr-D": result["metrics"]["CIDEr-D"],
                "generation_seconds": result["generation_seconds"],
            }
        )
    with EPOCH_HISTORY_CSV.open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=list(rows[0])
        )
        writer.writeheader()
        writer.writerows(rows)


optimizer, scheduler, scaler = build_optimizer_state()
progress = {
    "epoch": 0,
    "cursor": 0,
    "global_optimizer_step": 0,
    "epoch_accumulators": {
        "human_loss_sum": 0.0,
        "sequence_loss_sum": 0.0,
        "kd_loss_sum": 0.0,
        "human_tokens": 0,
        "sequence_tokens": 0,
        "kd_positions": 0,
        "overflow_retries": 0,
    },
}
history = []
best_result = None
no_improve_epochs = 0
resume_source = None

if LAST_CHECKPOINT.is_file():
    resume_source = LAST_CHECKPOINT
elif INPUT_RESUME_CHECKPOINT is not None:
    resume_source = INPUT_RESUME_CHECKPOINT
    previous_root = resume_source.parent.parent
    for previous_directory, current_directory in [
        (previous_root / "logs", LOG_DIR),
        (previous_root / "validation", VALIDATION_DIR),
    ]:
        if previous_directory.is_dir():
            for previous_file in previous_directory.glob("*"):
                if previous_file.is_file():
                    current_file = (
                        current_directory / previous_file.name
                    )
                    if not current_file.exists():
                        shutil.copy2(previous_file, current_file)
    previous_best = resume_source.parent / BEST_CHECKPOINT.name
    if previous_best.is_file() and not BEST_CHECKPOINT.exists():
        shutil.copy2(previous_best, BEST_CHECKPOINT)

if resume_source is not None:
    progress, history, best_result, no_improve_epochs = (
        load_checkpoint(
            resume_source, optimizer, scheduler, scaler
        )
    )
    print("Resumed from              :", resume_source)
    print("Resume progress          :", progress)
else:
    print("Starting from locked Student initialization")

# Establish a recoverable optimizer-step boundary before entering the loop.
# On manual interruption, restore this last known-good checkpoint instead of
# serializing a potentially half-computed microbatch.
save_checkpoint(
    LAST_CHECKPOINT,
    optimizer, scheduler, scaler,
    dict(progress), history, best_result,
    no_improve_epochs,
)

TRAINING_COMPLETED = False
EARLY_STOPPED = False
INTERRUPTED_SAFELY = False
training_started_at = time.perf_counter()
SESSION_BUDGET_SECONDS = 10.5 * 3600
SESSION_BUDGET_REACHED = False


class SessionBudgetReached(Exception):
    pass


try:
    while int(progress["epoch"]) < MAX_EPOCHS:
        epoch_index = int(progress["epoch"])
        epoch_number = epoch_index + 1
        cursor = int(progress["cursor"])
        loader = build_train_loader(epoch_index, cursor=cursor)
        iterator = iter(loader)
        accumulator = progress.get(
            "epoch_accumulators",
            {
                "human_loss_sum": 0.0,
                "sequence_loss_sum": 0.0,
                "kd_loss_sum": 0.0,
                "human_tokens": 0,
                "sequence_tokens": 0,
                "kd_positions": 0,
                "overflow_retries": 0,
            },
        )
        # Migrate checkpoints emitted by the first Hybrid draft.  They
        # can only have cursor=0 because execution previously stopped
        # before the first optimizer step.
        if "sequence_loss_sum" not in accumulator:
            assert cursor == 0
            accumulator["sequence_loss_sum"] = 0.0
        if "sequence_tokens" not in accumulator:
            assert cursor == 0
            accumulator["sequence_tokens"] = 0
        epoch_human_loss_sum = float(
            accumulator["human_loss_sum"]
        )
        epoch_sequence_loss_sum = float(
            accumulator["sequence_loss_sum"]
        )
        epoch_kd_loss_sum = float(
            accumulator["kd_loss_sum"]
        )
        epoch_human_tokens = int(accumulator["human_tokens"])
        epoch_sequence_tokens = int(
            accumulator["sequence_tokens"]
        )
        epoch_kd_positions = int(
            accumulator["kd_positions"]
        )
        epoch_images = cursor
        epoch_overflows = int(
            accumulator["overflow_retries"]
        )

        while True:
            microbatches = []
            for _ in range(ACCUMULATION):
                try:
                    microbatches.append(next(iterator))
                except StopIteration:
                    break
            if not microbatches:
                break
            result = optimizer_step(
                microbatches, optimizer, scheduler, scaler,
                retry_limit=10,
            )
            images = sum(
                len(batch["image_ids"]) for batch in microbatches
            )
            epoch_images += images
            progress["cursor"] = epoch_images
            progress["global_optimizer_step"] = (
                int(progress["global_optimizer_step"]) + 1
            )
            epoch_human_loss_sum += (
                result["human_ce"] * result["human_tokens"]
            )
            epoch_sequence_loss_sum += (
                result["sequence_ce"] * result["kd_positions"]
            )
            epoch_kd_loss_sum += (
                result["mapped_kl_t2"]
                * result["kd_positions"]
            )
            epoch_human_tokens += result["human_tokens"]
            epoch_sequence_tokens += result["kd_positions"]
            epoch_kd_positions += result["kd_positions"]
            epoch_overflows += result["overflow_retries"]
            progress["epoch_accumulators"] = {
                "human_loss_sum": epoch_human_loss_sum,
                "sequence_loss_sum": epoch_sequence_loss_sum,
                "kd_loss_sum": epoch_kd_loss_sum,
                "human_tokens": epoch_human_tokens,
                "sequence_tokens": epoch_sequence_tokens,
                "kd_positions": epoch_kd_positions,
                "overflow_retries": epoch_overflows,
            }
            step_row = {
                "created_at_utc": now_utc(),
                "epoch": epoch_number,
                "cursor": epoch_images,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
                **result,
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_adapter": optimizer.param_groups[1]["lr"],
                "lr_decoder": optimizer.param_groups[2]["lr"],
            }
            append_step_log(step_row)
            if (
                int(progress["global_optimizer_step"]) % 50 == 0
            ):
                print(
                    f"epoch={epoch_number:02d} | "
                    f"step={progress['global_optimizer_step']:05d} | "
                    f"cursor={epoch_images:06d} | "
                    f"loss={result['combined_loss']:.6f}"
                )
            if (
                int(progress["global_optimizer_step"]) % 250 == 0
            ):
                save_checkpoint(
                    LAST_CHECKPOINT,
                    optimizer, scheduler, scaler,
                    dict(progress), history, best_result,
                    no_improve_epochs,
                )
            if (
                time.perf_counter() - training_started_at
                >= SESSION_BUDGET_SECONDS
            ):
                save_checkpoint(
                    LAST_CHECKPOINT,
                    optimizer, scheduler, scaler,
                    dict(progress), history, best_result,
                    no_improve_epochs,
                )
                SESSION_BUDGET_REACHED = True
                raise SessionBudgetReached
        assert epoch_images == 113287
        assert epoch_human_tokens > 0
        assert epoch_sequence_tokens == epoch_kd_positions
        assert epoch_kd_positions > 0

        validation = evaluate_validation(epoch_number)
        validation.update(
            {
                "train_human_ce": (
                    epoch_human_loss_sum / epoch_human_tokens
                ),
                "train_sequence_ce": (
                    epoch_sequence_loss_sum / epoch_sequence_tokens
                ),
                "train_mapped_kl_t2": (
                    epoch_kd_loss_sum / epoch_kd_positions
                ),
                "train_combined_loss": (
                    ALPHA_HUMAN
                    * epoch_human_loss_sum / epoch_human_tokens
                    + ALPHA_SEQUENCE
                    * epoch_sequence_loss_sum / epoch_sequence_tokens
                    + ALPHA_KD
                    * epoch_kd_loss_sum / epoch_kd_positions
                ),
                "train_human_tokens": epoch_human_tokens,
                "train_sequence_tokens": epoch_sequence_tokens,
                "train_kd_positions": epoch_kd_positions,
                "train_images": epoch_images,
                "overflow_retries": epoch_overflows,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
            }
        )
        history.append(validation)

        previous_best_cider = (
            float(best_result["metrics"]["CIDEr-D"])
            if best_result is not None else float("-inf")
        )
        if (
            best_result is None
            or selection_key(validation) > selection_key(best_result)
        ):
            best_result = copy.deepcopy(validation)
            save_checkpoint(
                BEST_CHECKPOINT,
                optimizer, scheduler, scaler,
                {
                    "epoch": epoch_index + 1,
                    "cursor": 0,
                    "global_optimizer_step": progress[
                        "global_optimizer_step"
                    ],
                },
                history, best_result, no_improve_epochs,
            )

        if (
            float(validation["metrics"]["CIDEr-D"])
            >= previous_best_cider + 0.001
        ):
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        progress["epoch"] = epoch_index + 1
        progress["cursor"] = 0
        progress["epoch_accumulators"] = {
            "human_loss_sum": 0.0,
            "sequence_loss_sum": 0.0,
            "kd_loss_sum": 0.0,
            "human_tokens": 0,
            "sequence_tokens": 0,
            "kd_positions": 0,
            "overflow_retries": 0,
        }
        save_checkpoint(
            LAST_CHECKPOINT,
            optimizer, scheduler, scaler,
            dict(progress), history, best_result,
            no_improve_epochs,
        )
        write_epoch_history(history)
        print(
            f"epoch={epoch_number:02d} COMPLETE | "
            f"CIDEr-D={validation['metrics']['CIDEr-D']:.6f} | "
            f"best={best_result['metrics']['CIDEr-D']:.6f} | "
            f"CE={validation['teacher_forced_ce']:.6f}"
        )
        if (
            epoch_number >= 5
            and no_improve_epochs >= 4
        ):
            EARLY_STOPPED = True
            break

    TRAINING_COMPLETED = True
except (KeyboardInterrupt, SessionBudgetReached):
    (
        progress,
        history,
        best_result,
        no_improve_epochs,
    ) = load_checkpoint(
        LAST_CHECKPOINT, optimizer, scheduler, scaler
    )
    INTERRUPTED_SAFELY = True
    print(
        "Session budget reached."
        if SESSION_BUDGET_REACHED
        else "Manual interruption captured."
    )
    print("Restored last verified optimizer-step boundary:", progress)

training_seconds_this_session = (
    time.perf_counter() - training_started_at
)
write_json(
    RUN_STATE_PATH,
    {
        "schema_version": "xlkd05_run_state_v1",
        "created_at_utc": now_utc(),
        "status": (
            "complete" if TRAINING_COMPLETED
            else "interrupted_resume_ready"
        ),
        "progress": progress,
        "history_epochs": len(history),
        "best_result": best_result,
        "early_stopped": EARLY_STOPPED,
        "interrupted_safely": INTERRUPTED_SAFELY,
        "session_budget_reached": SESSION_BUDGET_REACHED,
        "training_seconds_this_session": training_seconds_this_session,
        "last_checkpoint": str(LAST_CHECKPOINT),
        "last_checkpoint_sha256": (
            sha256_file(LAST_CHECKPOINT)
            if LAST_CHECKPOINT.is_file() else None
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("XLKD-05 CELL 5 STATUS")
print("=" * 72)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Global optimizer steps    :", progress["global_optimizer_step"])
print("Current cursor            :", progress["cursor"])
print("Best epoch                :", (
    best_result["epoch"] if best_result else None
))
print("Best CIDEr-D              :", (
    best_result["metrics"]["CIDEr-D"] if best_result else None
))
print("Early stopped             :", EARLY_STOPPED)
print("Interrupted safely        :", INTERRUPTED_SAFELY)
print("Session budget reached    :", SESSION_BUDGET_REACHED)
print("Last checkpoint           :", LAST_CHECKPOINT)
print("Test Set status           : SEALED")
print("Next                      : package complete or resume state")


Resumed from              : /kaggle/input/notebooks/dngynn/hyb-123/HYB_123/checkpoints/hyb_s1_seed123_last.pt
Resume progress          : {'epoch': 12, 'cursor': 32320, 'global_optimizer_step': 21757, 'epoch_accumulators': {'human_loss_sum': 942290.4923658818, 'sequence_loss_sum': 177060.95059702918, 'kd_loss_sum': 794960.1220402718, 'human_tokens': 370184, 'sequence_tokens': 339854, 'kd_positions': 339854, 'overflow_retries': 0}}
epoch=13 | step=21800 | cursor=035072 | loss=1.971871
epoch=13 | step=21850 | cursor=038272 | loss=1.994804
epoch=13 | step=21900 | cursor=041472 | loss=2.059195
epoch=13 | step=21950 | cursor=044672 | loss=1.954339
epoch=13 | step=22000 | cursor=047872 | loss=1.847970
epoch=13 | step=22050 | cursor=051072 | loss=1.908954
epoch=13 | step=22100 | cursor=054272 | loss=2.011294
epoch=13 | step=22150 | cursor=057472 | loss=2.006787
epoch=13 | step=22200 | cursor=060672 | loss=2.033499
epoch=13 | step=22250 | cursor=063872 | loss=2.047778
epoch=13 | step=22300 | cu

PTBTokenizer tokenized 307821 tokens at 670252.04 tokens per second.
PTBTokenizer tokenized 51851 tokens at 282359.67 tokens per second.


{'testlen': 46852, 'reflen': 47088, 'guess': [46852, 41852, 36852, 31852], 'correct': [35673, 19871, 9646, 4523]}
ratio: 0.9949881073734074
epoch=13 COMPLETE | CIDEr-D=1.090417 | best=1.091766 | CE=2.768698
XLKD-05 CELL 5 STATUS
Status                    : COMPLETE
Completed epochs          : 13
Global optimizer steps    : 23023
Current cursor            : 0
Best epoch                : 9
Best CIDEr-D              : 1.0917658467011184
Early stopped             : True
Interrupted safely        : False
Session budget reached    : False
Last checkpoint           : /kaggle/working/HYB_123/checkpoints/hyb_s1_seed123_last.pt
Test Set status           : SEALED
Next                      : package complete or resume state


## Cell 6 — Package confirmation seed 123

Ghi complete/resume marker, comparison với baseline S1 cùng seed, manifest
SHA-256 và trạng thái sẵn sàng đưa vào XLKD-06 three-seed aggregate.


In [6]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "xlkd05_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "xlkd05_manifest.json"
SHA256SUMS = MANIFEST_DIR / "HYB_SHA256SUMS.txt"

if TRAINING_COMPLETED:
    assert best_result is not None
    assert BEST_CHECKPOINT.is_file()
    assert LAST_CHECKPOINT.is_file()
    kd_metrics = {
        key: float(value)
        for key, value in best_result["metrics"].items()
    }
    comparison = {
        "schema_version": "hyb_seed123_comparison_v1",
        "created_at_utc": now(),
        "baseline_seed123": {
            **BASELINE_METRICS,
            "CE": BASELINE_CE,
        },
        "hybrid_kd_seed123": {
            **kd_metrics,
            "CE": float(best_result["teacher_forced_ce"]),
            "best_epoch": int(best_result["epoch"]),
        },
        "delta_kd_minus_baseline": {
            **{
                key: kd_metrics[key] - BASELINE_METRICS[key]
                for key in BASELINE_METRICS
            },
            "CE": float(best_result["teacher_forced_ce"]) - BASELINE_CE,
        },
        "selection_primary": "CIDEr-D",
        "selection_tiebreakers": ["BLEU-4", "lower CE", "earlier epoch"],
        "kd_improves_cider_by_0_001": (
            kd_metrics["CIDEr-D"] >= BASELINE_CIDER + 0.001
        ),
        "fairness_disclosure": (
            "Post-baseline KD fine-tuning: 20 baseline epochs plus up to "
            "20 KD epochs; total training budget is not equal to baseline."
        ),
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    COMPARISON_PATH = (
        RESULT_DIR / "HYB_123_vs_S1.json"
    )
    write_json(COMPARISON_PATH, comparison)
    FINAL_MARKER = OUTPUT_ROOT / (
        "HYB_123_COMPLETE.json"
    )
else:
    assert LAST_CHECKPOINT.is_file()
    comparison = None
    FINAL_MARKER = OUTPUT_ROOT / (
        "HYB_123_RESUME_READY.json"
    )

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(MANIFEST_JSON, {
    "schema_version": "xlkd05_manifest_v1",
    "created_at_utc": now(),
    "artifact_count": len(manifest_rows),
    "artifacts": manifest_rows,
})
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

marker_payload = {
    "schema_version": (
        "xlkd05_seed123_full_run_complete_v1"
        if TRAINING_COMPLETED
        else "xlkd05_seed123_resume_ready_v1"
    ),
    "created_at_utc": now(),
    "status": "complete" if TRAINING_COMPLETED else "resume_ready",
    "experiment_id": "EXPANSIONNET_S1_HYBRID_KD",
    "method": "Hybrid Sequence + Dense Mapped Logit-KD",
    "seed": SEED,
    "temperature": TEMPERATURE,
    "alpha_human_ce": ALPHA_HUMAN,
    "alpha_sequence_ce": ALPHA_SEQUENCE,
    "alpha_kd": ALPHA_KD,
    "progress": progress,
    "completed_epochs": len(history),
    "best_result": best_result,
    "last_checkpoint": {
        "relative_path": LAST_CHECKPOINT.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256(LAST_CHECKPOINT),
    },
    "best_checkpoint": (
        {
            "relative_path": BEST_CHECKPOINT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256(BEST_CHECKPOINT),
        }
        if BEST_CHECKPOINT.is_file() else None
    ),
    "comparison": comparison,
    "confirmation_run": True,
    "ready_for_xlkd06_aggregate": bool(TRAINING_COMPLETED),
    "production_contract": {
        "source_sha256": sha256(PRODUCTION_CONTRACT_PATH),
        "status": "locked",
    },
    "fairness_disclosure": (
        "Post-baseline KD fine-tuning; report additional GPU hours and "
        "do not claim equal training budget."
    ),
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256(MANIFEST_CSV),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256(SHA256SUMS),
        },
    },
    "manifest_rows": len(manifest_rows),
    "manifest_verification": "all_passed",
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(FINAL_MARKER, marker_payload)

print("=" * 78)
print("XLKD-05 CELL 6 — PACKAGE")
print("=" * 78)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Last checkpoint           : VERIFIED")
print("Best checkpoint           :", BEST_CHECKPOINT.is_file())
print(
    "XLKD-06 aggregate         :",
    "READY" if TRAINING_COMPLETED else "NOT YET",
)
print("Validation selection      : CIDEr-D")
print("Test Set status           : SEALED")
print("Marker                    :", FINAL_MARKER)
print(
    "Next                      :",
    (
        "Complete both XLKD-05 seeds, then run XLKD-06 aggregate"
        if TRAINING_COMPLETED
        else "Save Version, attach this output, rerun XLKD-05 seed 123"
    ),
)


XLKD-05 CELL 6 — PACKAGE
Status                    : COMPLETE
Completed epochs          : 13
Manifest rows             : 40
Manifest verification     : ALL PASSED
Last checkpoint           : VERIFIED
Best checkpoint           : True
XLKD-06 aggregate         : READY
Validation selection      : CIDEr-D
Test Set status           : SEALED
Marker                    : /kaggle/working/HYB_123/HYB_123_COMPLETE.json
Next                      : Complete both XLKD-05 seeds, then run XLKD-06 aggregate
